In [15]:
!nvidia-smi


Sun May  3 03:50:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             33W /   70W |       3MiB /  15360MiB |      4%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
import os
import shutil


# Set up working directory and unzip the project
os.makedirs('/content/work', exist_ok=True)
%cd /content/work
!unzip -q -o /content/drive/MyDrive/research_for_colab.zip -d .

# Restructure into the expected layout: research/{src, data/processed, requirements.txt}
# (PowerShell's Compress-Archive flattens leaf paths, so we rebuild the structure here.)
os.makedirs('/content/work/research/data', exist_ok=True)

for src, dst in [
    ('/content/work/processed',        '/content/work/research/data/processed'),
    ('/content/work/src',              '/content/work/research/src'),
    ('/content/work/requirements.txt', '/content/work/research/requirements.txt'),
]:
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.move(src, dst)

%cd /content/work/research
!ls
!ls data/processed/

/content/work
/content/work/research
data  requirements.txt	results  src
full_train.csv	test.csv  train.csv  val.csv


In [4]:
!pip install -q transformers datasets accelerate sentencepiece scikit-learn emoji
import torch
print('CUDA:', torch.cuda.is_available(), '| Torch:', torch.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 14.3 MB/s eta 0:00:00
CUDA: True | Torch: 2.10.0+cu128


In [16]:
import subprocess

# IndicBERT seeds 123 and 2024 (seed 42 already done)
for seed in [123, 2024]:
    print(f"\n{'='*60}")
    print(f"=== IndicBERT v2, seed={seed} ===")
    print('='*60)
    subprocess.run([
        "python", "src/train_transformer.py",
        "--model", "ai4bharat/IndicBERTv2-MLM-only",
        "--seed", str(seed),
        "--epochs", "5",
        "--batch_size", "32",
    ], check=True)



=== IndicBERT v2, seed=123 ===

=== IndicBERT v2, seed=2024 ===


In [17]:
# All three HingRoBERTa seeds
for seed in [42, 123, 2024]:
    print(f"\n{'='*60}")
    print(f"=== HingRoBERTa, seed={seed} ===")
    print('='*60)
    subprocess.run([
        "python", "src/train_transformer.py",
        "--model", "l3cube-pune/hing-roberta",
        "--seed", str(seed),
        "--epochs", "5",
        "--batch_size", "32",
    ], check=True)



=== HingRoBERTa, seed=42 ===

=== HingRoBERTa, seed=123 ===

=== HingRoBERTa, seed=2024 ===


In [18]:
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M')
!cp -r results /content/drive/MyDrive/colab_results_{ts}
print(f"\nDone. Saved to: /content/drive/MyDrive/colab_results_{ts}")


Done. Saved to: /content/drive/MyDrive/colab_results_20260503_0437
